<h3>구조 청킹과 시맨틱 청킹</h3>
<p>먼저 Markdown 제목을 메타데이터로 보존합니다. 그다음 LangChain의 시맨틱 스플리터를 기본 설정으로 실행하고, 문장 사이 거리 중 어느 정도로 큰 값을 경계로 삼을지 정하는 백분위 값 하나만 바꿉니다.</p>

In [1]:
# 구조 스플리터, 시맨틱 스플리터, OpenAI 임베딩을 같은 커널에 설치합니다.
%pip install -q "python-dotenv==1.2.3" "langchain-openai==1.6.0"     "langchain-text-splitters==1.1.2" "langchain-experimental==0.4.2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter

# find_dotenv()는 상위 폴더를 탐색해 프로젝트의 .env 파일을 찾습니다.
# 현재 노트북 폴더부터 상위 폴더에서 키 파일을 찾습니다.
import os

env_file = find_dotenv(usecwd=True)
assert env_file, "실습 최상위 폴더에 .env를 준비하세요."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY를 확인하세요."
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=256,
)

<h3>1단계: Markdown 제목 구조 보존하기</h3>
<p>제목 기호와 메타데이터 이름을 연결합니다. 결과의 <code>metadata</code>와 <code>page_content</code>를 함께 확인합니다.</p>

In [3]:
markdown_text = """# 배송 정책
## 기본 배송
결제 다음 날 출고합니다.

## 제주 배송
기본 배송보다 이틀 더 걸립니다.

# 환불 정책
수령 후 7일 이내 신청합니다."""

# 튜플의 첫 값은 제목 기호이고, 둘째 값은 저장할 메타데이터 이름입니다.
headers = [("#", "section"), ("##", "topic")]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers)
structured_documents = header_splitter.split_text(markdown_text)

In [4]:
# Document 객체는 본문(page_content)과 부가 정보(metadata)를 함께 보관합니다.
for index, document in enumerate(structured_documents, start=1):
    print(f"[{index}] metadata={document.metadata}")
    print(document.page_content)
    print()

[1] metadata={'section': '배송 정책', 'topic': '기본 배송'}
결제 다음 날 출고합니다.

[2] metadata={'section': '배송 정책', 'topic': '제주 배송'}
기본 배송보다 이틀 더 걸립니다.

[3] metadata={'section': '환불 정책'}
수령 후 7일 이내 신청합니다.



<h3>2단계: 시맨틱 청킹 기본 설정 실행하기</h3>
<p>아래 문장은 마침표 뒤에 공백이 있습니다. 이 예제의 기본 문장 분리 정규식은 <code>.?!</code> 뒤의 공백을 경계로 사용합니다.</p>
<p>이 절은 유지보수가 종료된 langchain-experimental 0.4.2의 SemanticChunker로 원리를 학습합니다. 해당 버전에 한정한 실습이며 새 운영 시스템의 기본 선택을 권장하는 예제가 아닙니다.</p>

In [5]:
# 유지보수 종료 안내는 위 설명에 명시합니다. 이 경고만 반복 출력을 생략합니다.
import warnings
warnings.filterwarnings(
    "ignore", message="`langchain-experimental` is being sunset.*",
    category=DeprecationWarning,
)
from langchain_experimental.text_splitter import SemanticChunker

semantic_text = """배송은 결제 다음 날 시작합니다. 제주 배송은 이틀이 더 걸립니다. 배송 현황은 주문 내역에서 확인합니다. 환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다. 환불 금액은 결제한 수단으로 지급합니다. 신규 회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다."""

# 기본 percentile 방식은 인접 문장 사이 거리의 상위 구간을 경계로 선택합니다.
default_splitter = SemanticChunker(embeddings)
default_documents = default_splitter.create_documents([semantic_text])

for index, document in enumerate(default_documents, start=1):
    print(f"[{index}] {document.page_content}")

[1] 배송은 결제 다음 날 시작합니다. 제주 배송은 이틀이 더 걸립니다. 배송 현황은 주문 내역에서 확인합니다. 환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.
[2] 환불 금액은 결제한 수단으로 지급합니다. 신규 회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다.


<h3>3단계: 임계값 하나만 바꾸기</h3>
<p>모델과 원문은 그대로 두고 백분위 값을 95에서 50으로 낮춥니다. 95는 거리값을 작은 순서로 놓았을 때 약 상위 5%의 큰 거리만 후보로 삼고, 50은 중앙값보다 큰 거리도 후보로 삼는 설정입니다. 표본이 작거나 같은 값이 있으면 경계 비율이 정확히 5%나 50%가 되지는 않습니다. 실제 청크 수와 본문을 비교합니다.</p>

In [6]:
# breakpoint_threshold_amount만 50으로 지정합니다.
sensitive_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=50,
)
sensitive_documents = sensitive_splitter.create_documents([semantic_text])

print("기본 설정 청크 수:", len(default_documents))
print("50 백분위 청크 수:", len(sensitive_documents))
for index, document in enumerate(sensitive_documents, start=1):
    print(f"[{index}] {document.page_content}")

기본 설정 청크 수: 2
50 백분위 청크 수: 4
[1] 배송은 결제 다음 날 시작합니다. 제주 배송은 이틀이 더 걸립니다. 배송 현황은 주문 내역에서 확인합니다. 환불은 수령 후 7일 이내 신청합니다.
[2] 사용한 상품은 환불할 수 없습니다.
[3] 환불 금액은 결제한 수단으로 지급합니다.
[4] 신규 회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다.


<h3>일부러 실패해 보기</h3>
<p>백분위 값을 0으로 낮춰 거의 모든 양의 거리값을 경계 후보로 만든 결과의 청크 수와 본문을 확인합니다. 이 예제에서는 대부분의 문장 경계가 선택되어 짧은 조각이 많아집니다.</p>

In [7]:
# 낮은 임계값에서 청크가 얼마나 잘게 나뉘는지 비교합니다.
over_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=0,
)
over_documents = over_splitter.create_documents([semantic_text])

sentence_count = 8
print("문장 수:", sentence_count)
print("청크 수:", len(over_documents))
print("0 백분위 청크 본문:")
for document in over_documents:
    print(document.page_content)
assert len(over_documents) > len(sensitive_documents)
print("50 백분위보다 청크가 더 잘게 나뉘었습니다.")

문장 수: 8
청크 수: 7
0 백분위 청크 본문:
배송은 결제 다음 날 시작합니다. 제주 배송은 이틀이 더 걸립니다.
배송 현황은 주문 내역에서 확인합니다.
환불은 수령 후 7일 이내 신청합니다.
사용한 상품은 환불할 수 없습니다.
환불 금액은 결제한 수단으로 지급합니다.
신규 회원 등급은 매월 갱신합니다.
포인트는 주문 금액의 1퍼센트가 적립됩니다.
50 백분위보다 청크가 더 잘게 나뉘었습니다.


<p><strong>수정 방법</strong> 거리값의 중앙을 기준으로 삼는 50 백분위 결과를 사용하고, 0 백분위 결과보다 청크 수가 줄었는지 검증하고 본문이 어떻게 합쳐졌는지 확인합니다.</p>

In [8]:
# 앞에서 만든 sensitive_documents는 같은 원문에 50 백분위를 적용한 결과입니다.
assert 1 < len(sensitive_documents) < len(over_documents)
print("수정한 임계값:", 50)
print("청크 수:", len(sensitive_documents))
print("모든 문장을 각각 분리하지 않았습니다:", True)

수정한 임계값: 50
청크 수: 4
모든 문장을 각각 분리하지 않았습니다: True
